# Klasifikasi Fakta/Hoaks berbasis Kalimat & Ekstraksi Entitas (NER)

Arsitektur ini berfokus pada efisiensi tinggi menggunakan Google Colab GPU T4.
Terdiri dari dua arsitektur paralel:
1.  **IndoBERT Fine-Tuned (Trainer)**: Mendeteksi teks pada level *Sentence Classification*.
2.  **NER Pipeline Eksternal**: Menangkap entitas nama/lokasi pada kalimat untuk melengkapi konteks deteksi hoaks.
*Catatan: Augmentasi minoritas hanya diterapkan pada set latih (train-only), tidak pada validasi/uji.*


In [1]:
# 1. Instalasi Dependensi Inti
!pip install -q transformers datasets accelerate sentencepiece scikit-learn nltk

In [2]:
# 2. Pengaitan (Mounting) Google Drive
# Wajib dijalankan untuk mendapatkan akses ke folder scrape_output
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 3. Impor Library dan Persiapan Lingkungan GPU
import os
import gc
import shutil
import warnings
import re
import random
import json
import subprocess
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
    pipeline,
)

import nltk
from nltk.tokenize import sent_tokenize
from google.colab import files

warnings.filterwarnings("ignore")
nltk.download("punkt")
nltk.download("punkt_tab")

# Mematikan thread konversi safetensors otomatis di latar belakang
os.environ["HF_HUB_DISABLE_IMPLICIT_SAFETENSORS"] = "1"

print("Library dan Environment berhasil diinisialisasi.")



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Library dan Environment berhasil diinisialisasi.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [4]:
# 4. Konfigurasi Arsitektur (Dataset Processed + Training Konsisten Inference)
@dataclass
class Config:
    project_root: str = "/content"  # Sesuaikan jika Anda menjalankan di luar Colab

    # Raw CSV (dipakai oleh scripts/build_dataset.py)
    file_hoaks: str = "data_hoaks_turnbackhoaks.csv"
    file_nonhoaks_cnn: str = "data_nonhoaks_cnn.csv"
    file_nonhoaks_detik: str = "data_nonhoaks_detik.csv"
    file_nonhoaks_kompas: str = "data_nonhoaks_kompas.csv"

    # Processed dataset
    processed_dir: str = "data/processed"
    train_csv: str = "train.csv"
    val_csv: str = "val.csv"
    test_csv: str = "test.csv"
    leakage_audit_file: str = "leakage_audit.json"

    # Builder
    builder_script: str = "scripts/build_dataset.py"

    # Model
    model_name: str = "indolem/indobert-base-uncased"
    ner_model_name: str = "cahya/bert-base-indonesian-NER"

    # Training setup
    max_length: int = 192
    train_batch_size: int = 16
    eval_batch_size: int = 32
    grad_accumulation: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    num_epochs: int = 3
    auto_find_batch_size: bool = True
    early_stopping_patience: int = 2
    early_stopping_threshold: float = 0.0

    # Imbalance
    imbalance_method: str = "class_weight"  # opsi: class_weight | focal
    focal_gamma: float = 2.0

    # Label canonical
    label_fakta: int = 0
    label_hoaks: int = 1

    # Output
    output_dir: str = "indobert_hoax_ner_model_final"

    # Inference/Stress test
    orange_threshold: float = 0.65
    stress_samples_per_source: int = 2

    # Leakage guardrails
    leakage_gap_threshold: float = 0.20

    # Calibration
    calibration_threshold_min: float = 0.30
    calibration_threshold_max: float = 0.80
    calibration_threshold_step: float = 0.01
    calibration_min_hoax_recall: float = 0.70

    # Challenge
    challenge_min_correct: int = 5

    # NER inferensi
    ner_infer_device_default: int = -1
    ner_infer_use_gpu: bool = False
    ner_safe_offload_classifier: bool = True

    seed: int = 42

cfg = Config()
set_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.cuda.empty_cache()
    gc.collect()

ROOT = Path(cfg.project_root)
PROCESSED_DIR = ROOT / cfg.processed_dir
TRAIN_PATH = PROCESSED_DIR / cfg.train_csv
VAL_PATH = PROCESSED_DIR / cfg.val_csv
TEST_PATH = PROCESSED_DIR / cfg.test_csv
LEAKAGE_AUDIT_PATH = PROCESSED_DIR / cfg.leakage_audit_file
BUILDER_SCRIPT = ROOT / cfg.builder_script

print(f"Target komputasi: {device}")
print(f"Project root: {ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")



Target komputasi: cuda
Project root: /content
Processed dir: /content/data/processed


In [5]:
# 5. Data Loading dari Processed Dataset (fallback build_dataset.py)

TEXT_NORMALIZE_PATTERNS = [
    (re.compile(r"(?i)\buncategorized\b"), " "),
    (re.compile(r"(?i)\b(?:facebook|twitter|x\.com|tiktok|youtube|instagram|whatsapp)\b"), " "),
    (re.compile(r"(?i)\bakun\b[^.!?\n]{0,140}\bunggah\b[^.!?\n]*"), " "),
    (re.compile(r"(?i)\bbaca juga:\s*[^.!?\n]*"), " "),
    (re.compile(r"(?i)\blihat juga:\s*[^.!?\n]*"), " "),
    (re.compile(r"(?i)\badvertisement\b\s*scroll to continue with content"), " "),
    (re.compile(r"(?i)\bturnbackhoax(?:s)?\b"), " "),
    (re.compile(r"(?i)\bcnn indonesia\b"), " "),
    (re.compile(r"(?i)\bkompas\.com\b"), " "),
    (re.compile(r"(?i)\bdetik(?:com)?\b"), " "),
    (re.compile(r"(?i)\bmafindo\b"), " "),
    (re.compile(r"(?i)\b\d{1,2}\s*[/-]\s*\d{1,2}\s*[/-]\s*\d{2,4}\b"), " "),
    (re.compile(r"(?i)\b\d{1,2}\s+\d{1,2}\s+\d{4}\b"), " "),
    (re.compile(r"(?i)\b\d{1,2}:\d{2}\s*wib\b"), " "),
]

def normalize_unit_text(text: str) -> str:
    cleaned = str(text)
    for pattern, replacement in TEXT_NORMALIZE_PATTERNS:
        cleaned = pattern.sub(replacement, cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = cleaned.strip(" -:;,.")
    return cleaned


def assert_required_columns(df: pd.DataFrame, required: List[str], path: Path):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Kolom wajib tidak ditemukan di {path}: {missing}")


def load_processed_split(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"MISSING: {path}")
    df = pd.read_csv(path)
    assert_required_columns(df, ["text", "label", "source", "url_hash", "title_hash", "unit_type"], path)
    df = df.copy()
    df["text"] = df["text"].astype(str).map(normalize_unit_text)
    df = df[df["text"] != ""].copy()
    df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
    df = df[df["label"].isin([0, 1])].copy()
    df["label"] = df["label"].astype(int)
    return df


if not (TRAIN_PATH.exists() and VAL_PATH.exists() and TEST_PATH.exists() and LEAKAGE_AUDIT_PATH.exists()):
    print("Processed split/leakage audit belum lengkap. Menjalankan builder script...")
    if not BUILDER_SCRIPT.exists():
        raise FileNotFoundError(f"MISSING: {BUILDER_SCRIPT}")

    cmd = [
        sys.executable,
        str(BUILDER_SCRIPT),
        "--root",
        str(ROOT),
        "--output-dir",
        str(PROCESSED_DIR),
        "--leakage-gap-threshold",
        str(cfg.leakage_gap_threshold),
    ]
    print("Command:", " ".join(cmd))
    subprocess.run(cmd, check=True)


df_latih = load_processed_split(TRAIN_PATH)
df_validasi = load_processed_split(VAL_PATH)
df_uji = load_processed_split(TEST_PATH)

if not LEAKAGE_AUDIT_PATH.exists():
    raise FileNotFoundError(f"MISSING: {LEAKAGE_AUDIT_PATH}")

with open(LEAKAGE_AUDIT_PATH, "r", encoding="utf-8") as f:
    leakage_audit = json.load(f)

print("=== Ringkasan Processed Dataset ===")
for split_name, df_split in [("Latih", df_latih), ("Validasi", df_validasi), ("Uji", df_uji)]:
    print("")
    print(f"[{split_name}] rows={len(df_split)}")
    print(f"[{split_name}] distribusi label: {df_split['label'].value_counts().sort_index().to_dict()}")
    print(f"[{split_name}] source x label:")
    print(pd.crosstab(df_split["source"], df_split["label"]))

print("")
print("Contoh unit_type train:", df_latih["unit_type"].value_counts().to_dict())
print("Leakage audit:", {
    "pass": leakage_audit.get("pass"),
    "max_gap": leakage_audit.get("max_gap"),
    "threshold": leakage_audit.get("threshold"),
})

if not leakage_audit.get("pass", False):
    print("Top leakage markers:")
    for row in leakage_audit.get("worst_markers", [])[:5]:
        print(
            f"  - {row.get('marker')}: p0={row.get('label0_prevalence')} "
            f"p1={row.get('label1_prevalence')} gap={row.get('gap_abs')}"
        )
    raise RuntimeError(
        "Leakage audit FAIL. Jalankan perbaikan builder hingga marker-gap <= threshold sebelum training."
    )



=== Ringkasan Processed Dataset ===

[Latih] rows=39834
[Latih] distribusi label: {0: 24007, 1: 15827}
[Latih] source x label:
label             0      1
source                    
cnn             574      0
detik         13017      0
kompas         3558      0
turnbackhoax   6858  15827

[Validasi] rows=8441
[Validasi] distribusi label: {0: 5164, 1: 3277}
[Validasi] source x label:
label            0     1
source                  
cnn            125     0
detik         2835     0
kompas         760     0
turnbackhoax  1444  3277

[Uji] rows=8630
[Uji] distribusi label: {0: 5151, 1: 3479}
[Uji] source x label:
label            0     1
source                  
cnn            112     0
detik         2782     0
kompas         732     0
turnbackhoax  1525  3479

Contoh unit_type train: {'lead': 16926, 'claim': 8482, 'claim_title': 7345, 'debunk': 6858, 'hard_negative': 223}
Leakage audit: {'pass': True, 'max_gap': 0.040709, 'threshold': 0.2}


In [6]:
# 6. Tokenisasi Dataset Hugging Face
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)


def tokenisasi_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=cfg.max_length)


ds_latih = Dataset.from_pandas(df_latih, preserve_index=False)
ds_validasi = Dataset.from_pandas(df_validasi, preserve_index=False)
ds_uji = Dataset.from_pandas(df_uji, preserve_index=False)

ds_latih = ds_latih.map(tokenisasi_batch, batched=True)
ds_validasi = ds_validasi.map(tokenisasi_batch, batched=True)
ds_uji = ds_uji.map(tokenisasi_batch, batched=True)

kolom_format = ["input_ids", "attention_mask", "label"]
ds_latih.set_format(type="torch", columns=kolom_format)
ds_validasi.set_format(type="torch", columns=kolom_format)
ds_uji.set_format(type="torch", columns=kolom_format)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if device == "cuda" else None,
)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/39834 [00:00<?, ? examples/s]

Map:   0%|          | 0/8441 [00:00<?, ? examples/s]

Map:   0%|          | 0/8630 [00:00<?, ? examples/s]

In [7]:
# 7. Inisialisasi Model IndoBERT dan Pelatihan (Sinkron dengan Processed Dataset)
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=2,
    use_safetensors=False,
)
model.to(device)
model.config.use_cache = False
model.config.id2label = {0: "Fakta", 1: "Hoaks"}
model.config.label2id = {"Fakta": 0, "Hoaks": 1}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(p_weighted),
        "recall_weighted": float(r_weighted),
        "f1_weighted": float(f1_weighted),
        "cm_tn": int(tn),
        "cm_fp": int(fp),
        "cm_fn": int(fn),
        "cm_tp": int(tp),
    }


train_counts = df_latih["label"].value_counts().sort_index()
n_train = float(len(df_latih))
num_classes = 2
class_weights = []
for class_id in range(num_classes):
    count_class = float(train_counts.get(class_id, 1.0))
    class_weights.append(n_train / (num_classes * count_class))

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
if device == "cuda":
    class_weights_tensor = class_weights_tensor.to(device)

print(f"Class weights (0,1): {class_weights}")
print(f"Imbalance method: {cfg.imbalance_method}")


class ImbalanceTrainer(Trainer):
    def __init__(self, *args, class_weights=None, cfg_obj=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.cfg_obj = cfg_obj

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.cfg_obj.imbalance_method == "focal":
            ce_loss = F.cross_entropy(logits, labels, weight=self.class_weights, reduction="none")
            pt = torch.exp(-ce_loss)
            loss = ((1 - pt) ** self.cfg_obj.focal_gamma * ce_loss).mean()
        else:
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)

        return (loss, outputs) if return_outputs else loss


argumen_kwargs = dict(
    output_dir=cfg.output_dir,
    overwrite_output_dir=True,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accumulation,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    num_train_epochs=cfg.num_epochs,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    resume_from_checkpoint=None,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True if device == "cuda" else False,
    gradient_checkpointing=True,
    auto_find_batch_size=cfg.auto_find_batch_size,
    dataloader_num_workers=2,
    report_to="none",
)

training_arg_names = TrainingArguments.__init__.__code__.co_varnames
if "eval_strategy" in training_arg_names:
    argumen_kwargs["eval_strategy"] = "epoch"
else:
    argumen_kwargs["evaluation_strategy"] = "epoch"

argumen_latih = TrainingArguments(**argumen_kwargs)

trainer_kwargs = dict(
    model=model,
    args=argumen_latih,
    train_dataset=ds_latih,
    eval_dataset=ds_validasi,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    cfg_obj=cfg,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=cfg.early_stopping_patience,
            early_stopping_threshold=cfg.early_stopping_threshold,
        )
    ],
)

trainer_arg_names = Trainer.__init__.__code__.co_varnames
if "processing_class" in trainer_arg_names:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = ImbalanceTrainer(**trainer_kwargs)

print("Memulai pelatihan (processed dataset + metrik lengkap)...")
trainer.train(resume_from_checkpoint=None)

metric_validasi = trainer.evaluate(eval_dataset=ds_validasi)
if "eval_f1_macro" not in metric_validasi:
    raise RuntimeError(
        f"Metric eval_f1_macro tidak ditemukan pada hasil validasi: {sorted(metric_validasi.keys())}"
    )
if trainer.state.best_metric is None:
    raise RuntimeError("best_metric kosong; early stopping/selection metric tidak aktif dengan benar.")
metric_uji = trainer.evaluate(eval_dataset=ds_uji, metric_key_prefix="test")

print("")
print("=== Metrik Validasi ===")
for k, v in metric_validasi.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

print("")
print("=== Metrik Uji ===")
for k, v in metric_uji.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

# Calibration threshold pada validasi
pred_validasi = trainer.predict(ds_validasi)
logits_val = pred_validasi.predictions
labels_val = pred_validasi.label_ids

exp_val = np.exp(logits_val - logits_val.max(axis=1, keepdims=True))
probs_val = exp_val / exp_val.sum(axis=1, keepdims=True)
prob_hoax_val = probs_val[:, cfg.label_hoaks]

thresholds = np.arange(
    cfg.calibration_threshold_min,
    cfg.calibration_threshold_max + 1e-12,
    cfg.calibration_threshold_step,
)

candidate_rows = []
for thr in thresholds:
    pred_thr = (prob_hoax_val >= thr).astype(int)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels_val, pred_thr, average="macro", zero_division=0
    )
    p_each, r_each, f_each, _ = precision_recall_fscore_support(
        labels_val, pred_thr, labels=[0, 1], average=None, zero_division=0
    )
    cm = confusion_matrix(labels_val, pred_thr, labels=[0, 1])
    candidate_rows.append(
        {
            "threshold": float(round(float(thr), 4)),
            "f1_macro": float(f1_macro),
            "precision_macro": float(p_macro),
            "recall_macro": float(r_macro),
            "hoax_precision": float(p_each[1]) if len(p_each) > 1 else 0.0,
            "hoax_recall": float(r_each[1]) if len(r_each) > 1 else 0.0,
            "hoax_f1": float(f_each[1]) if len(f_each) > 1 else 0.0,
            "tn": int(cm[0, 0]),
            "fp": int(cm[0, 1]),
            "fn": int(cm[1, 0]),
            "tp": int(cm[1, 1]),
        }
    )

eligible = [r for r in candidate_rows if r["hoax_recall"] >= cfg.calibration_min_hoax_recall]
if eligible:
    best_row = sorted(eligible, key=lambda x: (x["f1_macro"], x["hoax_recall"]), reverse=True)[0]
    constrained_used = True
else:
    best_row = sorted(candidate_rows, key=lambda x: (x["f1_macro"], x["hoax_recall"]), reverse=True)[0]
    constrained_used = False

best_threshold = float(best_row["threshold"])
print("\n=== Calibration Threshold ===")
print(f"best_threshold={best_threshold:.4f} | constrained_used={constrained_used}")
print(best_row)
if not constrained_used:
    print("WARNING: Tidak ada threshold yang memenuhi batas minimal hoax recall.")

# Logging margin logit pada set uji
pred_uji = trainer.predict(ds_uji)
logits_uji = pred_uji.predictions
label_pred = np.argmax(logits_uji, axis=-1)
margin = np.abs(logits_uji[:, 1] - logits_uji[:, 0])
calibration_info = {
    "best_threshold": float(best_threshold),
    "constraint_recall_hoax": float(cfg.calibration_min_hoax_recall),
    "constrained_threshold_used": bool(constrained_used),
    "val_metrics_at_best": best_row,
    "logit_margin_mean": float(np.mean(margin)),
    "logit_margin_median": float(np.median(margin)),
    "logit_margin_p10": float(np.quantile(margin, 0.10)),
    "logit_margin_p90": float(np.quantile(margin, 0.90)),
    "pred_label_distribution_test_argmax": {
        "0": int((label_pred == 0).sum()),
        "1": int((label_pred == 1).sum()),
    },
}

os.makedirs(cfg.output_dir, exist_ok=True)
with open(os.path.join(cfg.output_dir, "calibration.json"), "w", encoding="utf-8") as f:
    json.dump(calibration_info, f, ensure_ascii=False, indent=2)

print("\n=== Calibration Artifact ===")
print(calibration_info)

# Pastikan mapping label semantik tersimpan
model.config.id2label = {0: "Fakta", 1: "Hoaks"}
model.config.label2id = {"Fakta": 0, "Hoaks": 1}
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

config_path = Path(cfg.output_dir) / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    saved_cfg = json.load(f)
saved_id2label = saved_cfg.get("id2label", {})
saved_label2id = saved_cfg.get("label2id", {})
if str(saved_id2label.get("0")) != "Fakta" or str(saved_id2label.get("1")) != "Hoaks":
    raise RuntimeError(f"Mapping id2label tidak sesuai di {config_path}: {saved_id2label}")
if int(saved_label2id.get("Fakta", -1)) != 0 or int(saved_label2id.get("Hoaks", -1)) != 1:
    raise RuntimeError(f"Mapping label2id tidak sesuai di {config_path}: {saved_label2id}")

with open(os.path.join(cfg.output_dir, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump({"id2label": {"0": "Fakta", "1": "Hoaks"}, "label2id": {"Fakta": 0, "Hoaks": 1}}, f, ensure_ascii=False, indent=2)

print(f"Model final tersimpan di: {cfg.output_dir}")
print(f"Verifikasi mapping sukses: {config_path}")




pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The tokenizer has new PAD

Class weights (0,1): [0.8296330237014204, 1.2584191571365388]
Imbalance method: class_weight
Memulai pelatihan (processed dataset + metrik lengkap)...


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 47, in spawn_con

...

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled



=== Metrik Validasi ===
eval_loss: 0.008581587113440037
eval_accuracy: 0.9986968368676697
eval_precision_macro: 0.99882395173743
eval_recall_macro: 0.9984331444636668
eval_f1_macro: 0.9986277483440233
eval_precision_weighted: 0.9986977730108451
eval_recall_weighted: 0.9986968368676697
eval_f1_weighted: 0.9986965815246572
eval_cm_tn: 5162
eval_cm_fp: 2
eval_cm_fn: 9
eval_cm_tp: 3268
eval_runtime: 13.484
eval_samples_per_second: 625.999
eval_steps_per_second: 19.579
epoch: 3.0

=== Metrik Uji ===
test_loss: 0.023045586422085762
test_accuracy: 0.996523754345307
test_precision_macro: 0.9968116662862847
test_recall_macro: 0.9959683217869493
test_f1_macro: 0.9963851283426044
test_precision_weighted: 0.9965298227468733
test_recall_weighted: 0.996523754345307
test_f1_weighted: 0.9965222778553374
test_cm_tn: 5145
test_cm_fp: 6
test_cm_fn: 24
test_cm_tp: 3455
test_runtime: 14.5463
test_samples_per_second: 593.276
test_steps_per_second: 18.561
epoch: 3.0

=== Calibration Threshold ===
best_thres


=== Calibration Artifact ===
{'best_threshold': 0.3, 'constraint_recall_hoax': 0.7, 'constrained_threshold_used': True, 'val_metrics_at_best': {'threshold': 0.3, 'f1_macro': 0.9987525683112568, 'precision_macro': 0.9989205887031394, 'recall_macro': 0.9985857230416344, 'hoax_precision': 0.9993885661877102, 'hoax_recall': 0.9975587427525175, 'hoax_f1': 0.9984728161270617, 'tn': 5162, 'fp': 2, 'fn': 8, 'tp': 3269}, 'logit_margin_mean': 10.672738075256348, 'logit_margin_median': 11.265625, 'logit_margin_p10': 9.546875, 'logit_margin_p90': 11.65625, 'pred_label_distribution_test_argmax': {'0': 5169, '1': 3461}}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model final tersimpan di: indobert_hoax_ner_model_final
Verifikasi mapping sukses: indobert_hoax_ner_model_final/config.json


In [8]:
# 8. Inference Konsisten + Stress Test Produksi
print("Mempersiapkan modul Named Entity Recognition (NER) sekunder...")

ner_device = cfg.ner_infer_device_default
device_klasifikasi = device

if cfg.ner_infer_use_gpu and torch.cuda.is_available():
    ner_device = 0
    if cfg.ner_safe_offload_classifier:
        model.to("cpu")
        device_klasifikasi = "cpu"
        torch.cuda.empty_cache()
        gc.collect()
        print("Model klasifikasi dipindah ke CPU sebelum memuat NER di GPU (mode aman).")
else:
    ner_device = -1

if ner_device >= 0 and not torch.cuda.is_available():
    ner_device = -1

ner_pipeline = pipeline(
    "ner",
    model=cfg.ner_model_name,
    aggregation_strategy="simple",
    device=ner_device,
)

HOAX_THRESHOLD = 0.5
CALIBRATION_LOADED = False
calibration_path = Path(cfg.output_dir) / "calibration.json"
if calibration_path.exists():
    with open(calibration_path, "r", encoding="utf-8") as f:
        calibration_payload = json.load(f)
    candidate_threshold = calibration_payload.get("best_threshold", calibration_payload.get("threshold"))
    if candidate_threshold is not None:
        HOAX_THRESHOLD = float(candidate_threshold)
        CALIBRATION_LOADED = True

print(f"Device klasifikasi: {device_klasifikasi} | Device NER pipeline: {ner_device}")
print(f"Hoax threshold: {HOAX_THRESHOLD:.4f} | calibration_loaded={CALIBRATION_LOADED}")

PARAGRAPH_SPLIT_RE = re.compile(r"(?:\r?\n){2,}")
SENTENCE_SPLIT_RE = re.compile(r'[^.!?]+(?:[.!?]+(?:["\)\]]+)?)|[^.!?]+$')


def split_paragraphs(text: str) -> List[str]:
    paragraphs = [p.strip() for p in PARAGRAPH_SPLIT_RE.split(str(text).strip()) if p.strip()]
    if paragraphs:
        return paragraphs
    raw = str(text).strip()
    return [raw] if raw else []


def split_sentences(paragraph: str) -> List[str]:
    normalized = normalize_unit_text(str(paragraph))
    if not normalized:
        return []
    sentences = [m.group(0).strip() for m in SENTENCE_SPLIT_RE.finditer(normalized)]
    return [s for s in sentences if s] or [normalized]


def classify_sentences(sentences: List[str]) -> List[Dict]:
    outputs = []
    model.eval()
    for s in sentences:
        unit_text = normalize_unit_text(s)
        if not unit_text:
            continue

        inputs = tokenizer(
            unit_text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.max_length,
            padding=True,
        ).to(device_klasifikasi)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).squeeze()
            argmax_id = int(torch.argmax(probs).item())
            prob_hoax = float(probs[cfg.label_hoaks].item())
            prob_fakta = float(probs[cfg.label_fakta].item())
            pred_id = int(1 if prob_hoax >= HOAX_THRESHOLD else 0)
            conf = max(prob_hoax, prob_fakta)
            label = "Hoaks" if pred_id == cfg.label_hoaks else "Fakta"

        color = "orange" if conf < cfg.orange_threshold else ("red" if label == "Hoaks" else "green")
        outputs.append(
            {
                "text": unit_text,
                "label": label,
                "pred_id": int(pred_id),
                "argmax_id": int(argmax_id),
                "prob_hoax": round(prob_hoax, 6),
                "prob_fakta": round(prob_fakta, 6),
                "confidence": round(conf, 6),
                "color": color,
            }
        )
    return outputs


def analyze_text_with_ner(text: str, include_ner: bool = True) -> Dict:
    paragraphs = split_paragraphs(text)
    analyzed_paragraphs = []
    all_sentences = []
    for p_idx, p in enumerate(paragraphs):
        sentences = split_sentences(p)
        cls = classify_sentences(sentences)
        analyzed_paragraphs.append({"paragraph_index": p_idx, "sentences": cls})
        all_sentences.extend(cls)

    ner_entities = []
    if include_ner:
        raw_ner = ner_pipeline(text)
        for ent in raw_ner:
            if float(ent.get("score", 0.0)) >= 0.60:
                ner_entities.append(
                    {
                        "word": ent.get("word", ""),
                        "entity_group": ent.get("entity_group", ""),
                        "score": round(float(ent.get("score", 0.0)), 6),
                    }
                )

    return {
        "paragraphs": analyzed_paragraphs,
        "summary": {
            "n_sentences": len(all_sentences),
            "n_hoaks": sum(1 for x in all_sentences if x["pred_id"] == cfg.label_hoaks),
            "n_fakta": sum(1 for x in all_sentences if x["pred_id"] == cfg.label_fakta),
            "hoax_threshold": float(HOAX_THRESHOLD),
        },
        "ner": ner_entities,
    }


print("\n=== STRESS TEST PRODUKSI ===")
if TEST_PATH.exists():
    df_stress = pd.read_csv(TEST_PATH)
    stress_rows = []
    for src in sorted(df_stress["source"].dropna().astype(str).unique()):
        subset = df_stress[df_stress["source"] == src]
        if subset.empty:
            continue
        stress_rows.append(subset.sample(n=min(cfg.stress_samples_per_source, len(subset)), random_state=cfg.seed))

    if stress_rows:
        stress_df = pd.concat(stress_rows, ignore_index=True)
        all_pred_ids = []
        for idx, row in stress_df.iterrows():
            text = str(row.get("text", ""))
            if not text.strip():
                continue
            result = analyze_text_with_ner(text, include_ner=False)
            sentence_preds = [s["pred_id"] for p in result["paragraphs"] for s in p["sentences"]]
            all_pred_ids.extend(sentence_preds)
            print("")
            print(
                f"[Stress {idx+1}] source={row.get('source')} true_label={row.get('label')} "
                f"pred_sentences={sentence_preds[:5]}"
            )
            if result["paragraphs"] and result["paragraphs"][0]["sentences"]:
                first = result["paragraphs"][0]["sentences"][0]
                print(
                    f"  contoh: pred_id={first['pred_id']} label={first['label']} "
                    f"prob_hoax={first['prob_hoax']} prob_fakta={first['prob_fakta']} conf={first['confidence']}"
                )

        if all_pred_ids:
            distribusi = pd.Series(all_pred_ids).value_counts().sort_index().to_dict()
            print("\nDistribusi pred_id stress test:", distribusi)
            if len(distribusi) < 2:
                print("WARNING: Prediksi stress test cenderung kolaps satu kelas.")
    else:
        print("Tidak ada data stress test yang bisa diambil dari processed test.")
else:
    print(f"MISSING: {TEST_PATH}")



Mempersiapkan modul Named Entity Recognition (NER) sekunder...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device klasifikasi: cuda | Device NER pipeline: -1
Hoax threshold: 0.3000 | calibration_loaded=True

=== STRESS TEST PRODUKSI ===

[Stress 1] source=cnn true_label=0 pred_sentences=[1, 0]
  contoh: pred_id=1 label=Hoaks prob_hoax=0.999603 prob_fakta=0.000397 conf=0.999603

[Stress 2] source=cnn true_label=0 pred_sentences=[0, 1]
  contoh: pred_id=0 label=Fakta prob_hoax=0.000276 prob_fakta=0.999724 conf=0.999724

[Stress 3] source=detik true_label=0 pred_sentences=[0, 0]
  contoh: pred_id=0 label=Fakta prob_hoax=2.8e-05 prob_fakta=0.999972 conf=0.999972

[Stress 4] source=detik true_label=0 pred_sentences=[0, 0]
  contoh: pred_id=0 label=Fakta prob_hoax=0.000553 prob_fakta=0.999447 conf=0.999447

[Stress 5] source=kompas true_label=0 pred_sentences=[0, 0]
  contoh: pred_id=0 label=Fakta prob_hoax=0.032314 prob_fakta=0.967686 conf=0.967686

[Stress 6] source=kompas true_label=0 pred_sentences=[0, 0]
  contoh: pred_id=0 label=Fakta prob_hoax=0.00051 prob_fakta=0.99949 conf=0.99949

[Stre

In [9]:
# 9. Uji Manual 6 Berita (Produksi)
berita_uji_6 = [
    """Gubernur DKI Jakarta Pramono Anung Wibowo disebut akan mengundang pihak-pihak terkait untuk membahas perizinan fasilitas padel di Jakarta. Langkah ini muncul setelah ada keluhan warga tentang kebisingan lapangan padel di dekat permukiman, dan pemerintah daerah disebut akan memetakan lokasi yang melanggar izin serta menyiapkan tindakan penertiban.""",
    """Beredar unggahan di media sosial yang mengklaim Kementerian Kehutanan membuka rekrutmen CPNS Polisi Kehutanan tahun 2026 secara besar-besaran. Unggahan itu menyebut kebutuhan personel sangat tinggi dan menyatakan lulusan SMA diprioritaskan, disertai ajakan agar publik segera mendaftar.""",
    """Sebuah pesawat pengangkut bahan bakar minyak dilaporkan jatuh di kawasan perbukitan Krayan, Kabupaten Nunukan, Kalimantan Utara. Laporan menyebut kepulan asap hitam terlihat setelah pesawat lepas landas usai mengantar pasokan BBM, dan saksi mata mengaku melihat pesawat sempat oleng sebelum jatuh pada kondisi cuaca yang berawan.""",
    """Beredar sebuah video yang memperlihatkan suasana bandara dengan aktivitas pesawat dan iring-iringan kendaraan mengantar tamu mancanegara. Video itu diklaim sebagai momen kedatangan rombongan Perserikatan Bangsa-Bangsa ke IKN Nusantara untuk agenda peresmian, dan narasi tersebut ramai dibagikan warganet.""",
    """PT Transjakarta dikabarkan melakukan modifikasi layanan pada empat rute mulai 21 Februari 2026, mencakup rute 4D, 6M, 9H, dan 11B. Perubahan ini disebut untuk mengoptimalkan layanan dan menjaga kenyamanan penumpang, dengan penyesuaian pada sebagian titik halte yang dilayani di rute tertentu.""",
    """Sebuah video di TikTok dipublikasikan dengan narasi breaking yang mengklaim terjadi kericuhan saat ribuan orang turun ke jalan untuk memprotes kebijakan Presiden AS Donald Trump. Unggahan tersebut memicu banyak komentar dan dibagikan ulang, dengan sebagian warganet menganggapnya sebagai rekaman aksi massa terbaru.""",
]
expected_labels = [0, 1, 0, 1, 0, 1]

print("=== UJI MANUAL 6 BERITA ===")
challenge_rows = []
for i, (teks, expected) in enumerate(zip(berita_uji_6, expected_labels), start=1):
    hasil = analyze_text_with_ner(teks, include_ner=True)
    semua_kalimat = [s for p in hasil["paragraphs"] for s in p["sentences"]]

    if semua_kalimat:
        prob_hoax_doc = float(max(x["prob_hoax"] for x in semua_kalimat))
        avg_hoax = float(np.mean([x["prob_hoax"] for x in semua_kalimat]))
        avg_fakta = float(np.mean([x["prob_fakta"] for x in semua_kalimat]))
        pred_label = int(1 if prob_hoax_doc >= HOAX_THRESHOLD else 0)
        label_doc = "Hoaks" if pred_label == 1 else "Fakta"
    else:
        prob_hoax_doc = 0.0
        avg_hoax = 0.0
        avg_fakta = 0.0
        pred_label = 0
        label_doc = "Fakta"

    row = {
        "berita_id": i,
        "expected": int(expected),
        "predicted": int(pred_label),
        "pass": bool(int(expected) == int(pred_label)),
        "prob_hoax_doc": round(prob_hoax_doc, 6),
        "avg_prob_hoax": round(avg_hoax, 6),
        "avg_prob_fakta": round(avg_fakta, 6),
        "n_sentences": int(len(semua_kalimat)),
        "n_hoaks": int(hasil["summary"]["n_hoaks"]),
        "n_fakta": int(hasil["summary"]["n_fakta"]),
        "label_doc": label_doc,
    }
    challenge_rows.append(row)

    print("")
    print(
        f"[Berita {i}] expected={expected} predicted={pred_label} ({label_doc}) "
        f"prob_hoax_doc={prob_hoax_doc:.4f} avg_hoax={avg_hoax:.4f} avg_fakta={avg_fakta:.4f}"
    )
    print(f"Ringkasan kalimat: {hasil['summary']}")
    if semua_kalimat:
        top = semua_kalimat[0]
        print(
            f"Kalimat contoh: pred_id={top['pred_id']} label={top['label']} "
            f"prob_hoax={top['prob_hoax']} prob_fakta={top['prob_fakta']} conf={top['confidence']}"
        )

    if hasil["ner"]:
        print("Entitas (maks 5):", hasil["ner"][:5])
    else:
        print("Entitas: tidak ada")

challenge_df = pd.DataFrame(challenge_rows)
print("\n=== Tabel Challenge 6 Berita ===")
print(challenge_df[["berita_id", "expected", "predicted", "pass", "prob_hoax_doc", "label_doc"]])

correct = int(challenge_df["pass"].sum())
pred_dist = challenge_df["predicted"].value_counts().sort_index().to_dict()
cm = confusion_matrix(challenge_df["expected"], challenge_df["predicted"], labels=[0, 1]).tolist()
print(f"\nAkurasi challenge: {correct}/{len(challenge_df)} = {correct/len(challenge_df):.4f}")
print("Confusion matrix [label0,label1]:", cm)
print("Distribusi prediksi challenge:", pred_dist)

if len(pred_dist) < 2:
    print("WARNING: Challenge kolaps ke satu kelas.")
if correct < cfg.challenge_min_correct:
    print(
        f"WARNING: Hasil challenge di bawah target minimum ({correct} < {cfg.challenge_min_correct})."
    )

challenge_payload = {
    "threshold": float(HOAX_THRESHOLD),
    "min_correct_target": int(cfg.challenge_min_correct),
    "correct": int(correct),
    "total": int(len(challenge_df)),
    "accuracy": float(correct / len(challenge_df)) if len(challenge_df) else 0.0,
    "pred_distribution": {str(k): int(v) for k, v in pred_dist.items()},
    "confusion_matrix": cm,
    "rows": challenge_rows,
}
challenge_path = Path(cfg.output_dir) / "challenge_eval.json"
with open(challenge_path, "w", encoding="utf-8") as f:
    json.dump(challenge_payload, f, ensure_ascii=False, indent=2)
print(f"Challenge eval saved: {challenge_path}")



=== UJI MANUAL 6 BERITA ===

[Berita 1] expected=0 predicted=0 (Fakta) prob_hoax_doc=0.0003 avg_hoax=0.0002 avg_fakta=0.9998
Ringkasan kalimat: {'n_sentences': 2, 'n_hoaks': 0, 'n_fakta': 2, 'hoax_threshold': 0.3}
Kalimat contoh: pred_id=0 label=Fakta prob_hoax=2.3e-05 prob_fakta=0.999977 conf=0.999977
Entitas (maks 5): [{'word': 'gubernur dki jakarta', 'entity_group': 'NOR', 'score': 0.991502}, {'word': 'pramono anung wibowo', 'entity_group': 'PER', 'score': 0.994731}, {'word': 'jakarta', 'entity_group': 'GPE', 'score': 0.997904}, {'word': 'lapangan padel', 'entity_group': 'LOC', 'score': 0.959229}, {'word': 'pemerintah daerah', 'entity_group': 'NOR', 'score': 0.915019}]

[Berita 2] expected=1 predicted=1 (Hoaks) prob_hoax_doc=0.9996 avg_hoax=0.4998 avg_fakta=0.5002
Ringkasan kalimat: {'n_sentences': 2, 'n_hoaks': 1, 'n_fakta': 1, 'hoax_threshold': 0.3}
Kalimat contoh: pred_id=0 label=Fakta prob_hoax=2.4e-05 prob_fakta=0.999976 conf=0.999976
Entitas (maks 5): [{'word': 'kementerian ke

In [10]:
drive.mount('/content/drive')
folder_tujuan = '/content/drive/MyDrive/berita_hoax_output'
os.makedirs(folder_tujuan, exist_ok=True)

nama_file_zip = "indobert_hoax_ner_model_final"
nama_file_zip_lengkap = f"{nama_file_zip}.zip"

print(f"Sedang mengompresi direktori {cfg.output_dir}...")
shutil.make_archive(nama_file_zip, 'zip', cfg.output_dir)

jalur_sumber = nama_file_zip_lengkap
jalur_target = os.path.join(folder_tujuan, nama_file_zip_lengkap)

print(f"Mengunggah file {nama_file_zip_lengkap} ke Google Drive di folder 'berita_hoax_output'...")
try:
    shutil.copy(jalur_sumber, jalur_target)
    print(f"Proses unggah berhasil. File tersimpan di: {jalur_target}")
except Exception as e:
    print(f"Terjadi kesalahan saat mengunggah ke Google Drive: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sedang mengompresi direktori indobert_hoax_ner_model_final...
Mengunggah file indobert_hoax_ner_model_final.zip ke Google Drive di folder 'berita_hoax_output'...
Proses unggah berhasil. File tersimpan di: /content/drive/MyDrive/berita_hoax_output/indobert_hoax_ner_model_final.zip


In [11]:
!pip install -q huggingface_hub

from huggingface_hub import login, HfApi
from google.colab import userdata

try:
    token_kredensial = userdata.get('HF_TOKEN')
    login(token=token_kredensial)
except Exception:
    print("Token HF_TOKEN tidak ditemukan pada Colab Secrets.")

api_huggingface = HfApi()
id_repositori_target = "fjrmhri/Deteksi_Hoax_TA"
direktori_sumber_artefak = cfg.output_dir

api_huggingface.create_repo(repo_id=id_repositori_target, private=False, exist_ok=True)

print(f"Memulai unggahan dari {direktori_sumber_artefak} menuju {id_repositori_target}...")
api_huggingface.upload_folder(
    folder_path=direktori_sumber_artefak,
    repo_id=id_repositori_target,
    repo_type="model",
    commit_message="Pembaruan model IndoBERT deteksi hoaks"
)
print(f"Transmisi selesai. URL Repositori: https://huggingface.co/{id_repositori_target}")

Memulai unggahan dari indobert_hoax_ner_model_final menuju fjrmhri/Deteksi_Hoax_TA...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-3735/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...kpoint-2490/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  ...nt-3735/model.safetensors:   1%|          | 3.92MB /  442MB            

  ...l_final/model.safetensors:   0%|          | 1.12MB /  442MB            

  ...ckpoint-2490/scheduler.pt:   7%|7         |   103B / 1.47kB            

  ...nt-2490/training_args.bin:   7%|7         |   365B / 5.20kB            

Transmisi selesai. URL Repositori: https://huggingface.co/fjrmhri/Deteksi_Hoax_TA
